In [1]:
import pandas as pd

In [2]:
rawdata = pd.read_csv("raw_data/cleaned_data.csv")

In [3]:
rawdata.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             1000 non-null   int64  
 1   device_type         1000 non-null   str    
 2   location            1000 non-null   str    
 3   age_group           1000 non-null   str    
 4   gender              1000 non-null   str    
 5   ad_id               1000 non-null   str    
 6   content_type        1000 non-null   str    
 7   ad_topic            1000 non-null   str    
 8   ad_target_audience  1000 non-null   str    
 9   click_through_rate  1000 non-null   float64
 10  conversion_rate     1000 non-null   float64
 11  engagement_level    1000 non-null   str    
 12  view_time           1000 non-null   int64  
 13  cost_per_click      1000 non-null   float64
 14  ROI                 1000 non-null   float64
 15  date                1000 non-null   str    
 16  time              

## Langchain

In [7]:
SYSTEM_PROMPT = """
You are an AI Assistant specialized in advertising campaign analytics.

You have access to a dataset with the following schema:

- user_id
- device_type
- location
- age_group
- gender
- ad_id
- content_type
- ad_topic
- ad_target_audience
- click_through_rate
- conversion_rate
- engagement_level
- view_time
- cost_per_click
- ROI
- ROI_Category
- date
- time

Primary role:
- Help users answer any dataset question with the right level of detail.
- Act like a practical assistant: concise for simple questions, deeper for analysis and optimization.
- When users ask for optimization, identify opportunities and recommend actions.

Capabilities:
1. Analyze campaign performance using Python
2. Explain results in plain language
3. Recommend optimization actions with rationale
4. Compare segments and prioritize by likely impact
5. Summarize findings clearly for decision-making

Response style rules:
- Adapt depth to user intent:
  - Simple query -> brief answer (2-5 lines).
  - Analytical or optimization query -> structured answer with evidence and actions.
- Do not over-explain. Do not be overly short. Match user intent.
- If request is ambiguous, ask one focused clarification question before deep analysis.
- If user message includes selected chart context, prioritize that chart in analysis.

CODE RULES:
- DataFrame name is `df`
- Use pandas, matplotlib (and seaborn/plotly when useful)
- Always include print statements
- Keep code minimal and clean

Analysis rules:
- NEVER assume values that are not in data.
- If data is required for the answer, ALWAYS use the `python` tool.
- For conceptual questions not requiring data access, answer directly.
- After tool output, always explain what the numbers imply.
- Mention uncertainty or data limitations when relevant.

Optimization workflow (when user asks to optimize):
1. Define objective metric(s): click_through_rate, conversion_rate, ROI, cost_per_click
2. Establish baseline performance
3. Segment by relevant dimensions (for example device_type, location, age_group, gender, content_type, ad_topic, ad_target_audience, time/date)
4. Find best and worst performing segments
5. Recommend top actions ranked by expected impact and feasibility
6. Suggest a simple validation plan (A/B test or holdout check)

Preferred answer format:
- Answer: direct response to the user question
- Evidence: key metrics/tables from analysis
- Actions: concrete next steps (only when relevant)

TOOL CALLING RULES (CRITICAL):
- The only available tool is named `python`.
- Tool arguments must be strict JSON object with this exact shape:
  {"code": "<python code as a string>"}
- Never send raw code as tool arguments.
- Never use ReAct-style text like "Action:" or "Observation:".
- Return only normal assistant text after tool results are available.

Be accurate, useful, and decision-oriented.
"""

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.agents import create_agent
import json
from dotenv import load_dotenv
import os

In [ ]:
# load_dotenv()
# _api_key = os.getenv("GROQ_API_KEY")
# print(_api_key)

In [3]:
llm = ChatGroq(
    api_key = _api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.3
)

In [10]:
from pathlib import Path
import subprocess
import json

def execute_python(code: str) -> dict:
    try:
        project_root = Path.cwd()  # ✅ FIXED for Jupyter

        data_path = project_root / "raw_data"
        csv_file_name = "cleaned_data.csv"
        host_csv_path = data_path / csv_file_name
        container_csv_path = f"/data/{csv_file_name}"

        plot_output_path = project_root / "tool_plot_output"
        plot_output_path.mkdir(parents=True, exist_ok=True)

        if not host_csv_path.exists():
            return {"status": "error", "error": "CSV not found", "stdout": ""}

        result = subprocess.run(
            [
                "docker", "run",
                "-i",
                "-e", f"CSV_PATH={container_csv_path}",
                "-v", f"{data_path}:/data:ro",
                "-v", f"{plot_output_path}:/tool_plot_output",
                "python_code_executor:v1"
            ],
            input=code,
            text=True,
            capture_output=True,
            timeout=20
        )

        if result.returncode != 0:
            return {"status": "error", "error": result.stderr, "stdout": result.stdout}

        parsed = json.loads(result.stdout)

        return parsed

    except Exception as e:
        return {"status": "error", "error": str(e), "stdout": ""}

In [11]:
from langchain.tools import tool

@tool("python")
def python_executor(code: str) -> str:
    """Execute python code in docker and return JSON result"""
    result = execute_python(code)
    return json.dumps(result)

In [ ]:
agent = create_agent(
    model=llm,
    tools=[python_executor],
    system_prompt=SYSTEM_PROMPT,
)

In [16]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is average ROI by device type?"}
    ]
})

print(response)

{'messages': [HumanMessage(content='What is average ROI by device type?', additional_kwargs={}, response_metadata={}, id='01cdac46-f938-4a92-b06e-71e659e50d5b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to compute average ROI by device_type. Use python.', 'tool_calls': [{'id': 'fc_0d0e532d-c762-472b-b44a-f6522b4a6472', 'function': {'arguments': '{"code":"import pandas as pd\\n# Assuming df is loaded\\navg_roi = df.groupby(\'device_type\')[\'ROI\'].mean().reset_index()\\nprint(avg_roi)"}', 'name': 'python'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 802, 'total_tokens': 869, 'completion_time': 0.071334331, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.00538772, 'prompt_tokens_details': {'cached_tokens': 768}, 'queue_time': 0.040277545, 'total_time': 0.076722051}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_r

In [18]:
import json

def parse_agent_response(response):
    structured = {
        "messages": [],
        "tool_calls": [],
        "tool_outputs": [],
        "final_answer": None,
        "usage": [],
        "reasoning": []
    }

    for msg in response["messages"]:
        base = {
            "id": getattr(msg, "id", None),
            "role": msg.type,
            "content": msg.content
        }

        # ---- Extract usage ----
        if hasattr(msg, "usage_metadata") and msg.usage_metadata:
            structured["usage"].append(msg.usage_metadata)

        # ---- Extract reasoning ----
        reasoning = msg.additional_kwargs.get("reasoning_content") if hasattr(msg, "additional_kwargs") else None
        if reasoning:
            structured["reasoning"].append(reasoning)

        # ---- Extract tool calls ----
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool in msg.tool_calls:
                structured["tool_calls"].append({
                    "id": tool.get("id"),
                    "name": tool.get("name"),
                    "arguments": tool.get("args")
                })

        # ---- Extract tool outputs ----
        if msg.type == "tool":
            try:
                parsed_output = json.loads(msg.content)
            except:
                parsed_output = msg.content

            structured["tool_outputs"].append({
                "tool_name": getattr(msg, "name", None),
                "tool_call_id": getattr(msg, "tool_call_id", None),
                "output": parsed_output
            })

        structured["messages"].append(base)

    # ---- Final Answer ----
    if structured["messages"]:
        structured["final_answer"] = structured["messages"][-1]["content"]

    return structured

In [19]:
parsed = parse_agent_response(response)

print(json.dumps(parsed, indent=2))

{
  "messages": [
    {
      "id": "01cdac46-f938-4a92-b06e-71e659e50d5b",
      "role": "human",
      "content": "What is average ROI by device type?"
    },
    {
      "id": "lc_run--019d761d-7075-7841-a074-ba97bfd9b1a9-0",
      "role": "ai",
      "content": ""
    },
    {
      "id": "d84d4e92-ca19-4b31-ade8-4c587eff33ca",
      "role": "tool",
      "content": "{\"status\": \"success\", \"stdout\": \"  device_type       ROI\\n0     Desktop  1.127606\\n1      Mobile  1.268846\\n2      Tablet  1.169218\\n\", \"error\": null, \"plot_path\": null}"
    },
    {
      "id": "lc_run--019d761d-7860-75d2-beff-cceec3f183ca-0",
      "role": "ai",
      "content": "**Answer:**  \nThe average ROI per device type is:\n\n| Device Type | Avg. ROI |\n|-------------|----------|\n| Desktop     | 1.13 |\n| Mobile      | 1.27 |\n| Tablet      | 1.17 |\n\n**Evidence:**  \nThe table above is derived from `df.groupby('device_type')['ROI'].mean()`.\n\n**Interpretation:**  \nMobile campaigns are del

## Langgraph

User Input
   ↓
[LLM Node] ── decides → tool call?
   ↓ yes                     ↓ no
[Tool Node]              [Final Answer]
   ↓
(back to LLM)

In [21]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool

In [33]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.3
)


In [34]:
@tool("python")
def python_executor(code: str) -> str:
    """Execute python code"""
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return json.dumps({"status": "success", "output": str(local_vars)})
    except Exception as e:
        return json.dumps({"status": "error", "error": str(e)})

In [35]:
llm_with_tools = llm.bind_tools([python_executor])

In [36]:
class AgentState(TypedDict):
    messages: List[Any]

In [37]:
def llm_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": state["messages"] + [response]
    }

In [38]:
def tool_node(state: AgentState):
    last_message = state["messages"][-1]

    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name == "python":
            result = python_executor.invoke(tool_args)

            tool_messages.append(
                ToolMessage(
                    content=result,
                    tool_call_id=tool_call["id"]
                )
            )

    return {
        "messages": state["messages"] + tool_messages
    }

In [39]:
def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool"

    return END

In [40]:
def build_graph():
    workflow = StateGraph(AgentState)

    workflow.add_node("llm", llm_node)
    workflow.add_node("tool", tool_node)

    workflow.set_entry_point("llm")

    workflow.add_conditional_edges(
        "llm",
        should_continue,
        {
            "tool": "tool",
            END: END
        }
    )

    workflow.add_edge("tool", "llm")

    return workflow.compile()

In [41]:
app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="What is average ROI by device type?")
    ]
})

In [42]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: What is average ROI by device type?

ai: I’d be happy to calculate that for you! Could you please share the data set (or at least the relevant columns such as device type, revenue, cost, etc.)? Once I have the numbers, I can compute the average ROI per device type.

